In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

In [5]:
IMAGE_DIR = Path.cwd().resolve() / "dataset/images"
LABEL_DIR = Path.cwd().resolve() / "dataset/masks"
DATASET_DIR = IMAGE_DIR / "GF1_WHU"
DATASET_NAME = "GF1_WFV1_E68.0_N57.9_20130705_L2A0000356197"
TEST_FEATURES = IMAGE_DIR / "GF1_WHU" / DATASET_NAME
TEST_LABELS = LABEL_DIR / "GF1_WHU" / DATASET_NAME

PREDICT_LABELS = Path("benchmark/predictions/")

assert TEST_FEATURES.exists()

AssertionError: 

In [ ]:
test_meta = pd.read_csv(DATASET_DIR / f"{DATASET_NAME}_metadata.csv")
test_meta.head()

In [ ]:
import xarray
import xrspatial.multispectral as ms

def get_xarray(filepath):
    """Put images in xarray.DataArray format"""
    im_arr = np.array(Image.open(filepath))
    return xarray.DataArray(im_arr, dims=["y", "x"])

def true_color_img(chip):
    """Given the path to the directory of Sentinel-2 chip feature images,
    plots the true color image"""
    red = get_xarray(chip.B04_path)
    green = get_xarray(chip.B03_path)
    blue = get_xarray(chip.B02_path)

    return ms.true_color(r=red, g=green, b=blue)

In [ ]:
random_chip = test_meta.sample(random_state=41).iloc[0]
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
im = true_color_img(random_chip)
ax[0].imshow(im)
ax[0].set_title(f"True color image for chip id {random_chip.chip_id}")
label_im = Image.open(TEST_LABELS / f"{random_chip.chip_id}.tif")
ax[1].imshow(label_im, cmap="viridis", vmin=0, vmax=2) # cmap="tab10"
ax[1].set_title(f"True chip {random_chip.chip_id} label")
label_im = Image.open(PREDICT_LABELS / f"{random_chip.chip_id}.tif")
ax[2].imshow(label_im, cmap="viridis", vmin=0, vmax=2)
ax[2].set_title(f"Predicted chip {random_chip.chip_id} label")
plt.tight_layout()
plt.show()

In [ ]:
import torch

ckpt_path = "checkpoints/epoch03-val_avg_iou0.8412.ckpt"
checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)

model_weights  = checkpoint["state_dict"]

torch.save(model_weights, "benchmark/unet/assets/cloud_model.pt")

In [ ]:
%%file main.py
import os
from pathlib import Path
from typing import List

from loguru import logger
import pandas as pd
from PIL import Image
import torch
import typer

from MyCloudSenseNet.benchmark.cloud_dataset import CloudDataset
from MyCloudSenseNet.benchmark.cloud_model import CloudModel

os.environ["TORCH_HOME"] = str("torch")

def get_metadata(features_dir: os.PathLike, bands: List[str]):
    chip_ids = (
        pth.name for pth in features_dir.iterdir() if not pth.name.startswith(".") and not pth.name.endswith(".csv")
    )
    rows = []
    for chip_id in chip_ids:
        row = {"chip_id": chip_id}
        for band in bands:
            row[f"{band}_path"] = features_dir / chip_id / f"{band}.tif"
        rows.append(row)
    return pd.DataFrame(rows)

def make_predictions(
    model: CloudModel,
    x_paths: pd.DataFrame,
    bands: List[str],
    predictions_dir: os.PathLike,
):
    test_dataset = CloudDataset(x_paths=x_paths, bands=bands)
    test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=model.batch_size,
        num_workers=model.num_workers,
        shuffle=False,
        pin_memory=True,
    )

    for batch_index, batch in enumerate(test_dataloader):
        logger.debug(f"Predicting batch {batch_index} of {len(test_dataloader)}")
        x = batch["chip"]
        preds = model.forward(x)
        preds = torch.softmax(preds, dim=1)[:, 1]
        preds = (preds > 0.5).detach().numpy().astype("uint8")
        for chip_id, pred in zip(batch["chip_id"], preds):
            chip_pred_path = predictions_dir / f"{chip_id}.tif"
            chip_pred_im = Image.fromarray(pred)
            chip_pred_im.save(chip_pred_path)

def main(
    model_weights_path: Path = "../cloud_model.pt",
    test_features_dir: Path = "../dataset/images/GF1_WFV1_E88.6_N28.0_20141208_L2A0000845193",
    predictions_dir: Path = "predictions",
    bands: List[str] = ["B02", "B03", "B04", "B08"],
    fast_dev_run: bool = False,
):
    if not test_features_dir.exists():
        raise ValueError(
            f"The directory for test feature images must exist and {test_features_dir} does not exist"
        )
    predictions_dir.mkdir(exist_ok=True, parents=True)

    logger.info("Loading model")
    model = CloudModel(bands=bands, hparams={"weights": None})
    model.load_state_dict(torch.load(model_weights_path))

    logger.info("Loading test metadata")
    test_metadata = get_metadata(test_features_dir, bands=bands)
    if fast_dev_run:
        test_metadata = test_metadata.head()
    logger.info(f"Found {len(test_metadata)} chips")

    logger.info("Generating predictions in batches")
    make_predictions(model, test_metadata, bands, predictions_dir)

    logger.info(f"""Saved {len(list(predictions_dir.glob("*.tif")))} predictions""")

if __name__ == "__main__":
    typer.run(main)